In [1]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import openai
print(openai.__version__)

1.99.5


In [4]:
from openai import OpenAI

# Замените 'api_key' на ваш реальный API ключ
api_key = "sk-sQel9sNQGifEDzy3Xf4N0A"

base_url = "http://litellm-proxy-ai-worker.app.kube02019.internal/v1/"

# Настройка клиента
# client = OpenAI(base_url="http://llm-core-ai-worker-llm.app.kube02018.internal/v1/", api_key=api_key)

# client = OpenAI(base_url="http://litellm-proxy-ai-worker.app.kube02019.internal/", api_key=api_key)
client = OpenAI(base_url=base_url, api_key=api_key)
# client = OpenAI(base_url="http://litellm-proxy-ai-worker.app.kube02019.internal/v1/models", api_key=api_key)

client

In [5]:
from openai import OpenAI

client = OpenAI(base_url=base_url, api_key=api_key)

# Получаем список доступных моделей
models = client.models.list()
print("Доступные модели:")
for model in models.data:
    print(f"- {model.id}")

Доступные модели:
- qwen2.5-vl-72b-instruct
- deepseek-r1
- qwen3-30b
- embedder
- qwen3-235b
- qwen3-coder-480b


In [6]:
models = [
    # "qwen2.5-vl-72b-instruct",
    "deepseek-r1",
    # "qwen3-30b",
    # "embedder",
    "qwen3-235b",
    "qwen3-coder-480b",
]

models

['deepseek-r1', 'qwen3-235b', 'qwen3-coder-480b']

# 1


In [ ]:
TEMPLATE = """ты - специалист-переводчик китайского и русского языков.
нужно:
сделать перевод китайского слова на русский язык.
перевод  должен быть средней длины, не более 20 слов.
в случае многозначных слов - пиши все варианты перевода, для таких слов - пусть перевод будет немного длиннее, главное чтобы были все значения.
в ответе напечатай только русский перевод.
переведи китайское слово: {word_foreign}.
"""

TEMPLATE

In [ ]:
!ls

In [ ]:
INPUT_FILE = "./words_Китайский_20250805_194526.json.0001_10000.txt"

INPUT_FILE

In [ ]:
INDEX_BEGIN = 14
INDEX_END = 10_000

INDEX_BEGIN, INDEX_END

In [ ]:
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    words = f.read().splitlines()

words = words[INDEX_BEGIN:INDEX_END]

print(words[0])
print(words[-1])


In [ ]:
result = {}

for entry in words:
    number, word = entry.split(". ")
    content = TEMPLATE.format(word_foreign=word)

    print("-"*20)
    print(number, word)

    for model in models:
        print("\t", model)
    
        try:
            chat_completion = client.chat.completions.create(
                messages=[
                    {
                        "role": "user",
                        "content": content,
                    }
                ],
                model=model,
                temperature=0.35,
                top_p=0.9
            )
            
            answer = chat_completion.choices[0].message.content
            answer = answer.replace("\n", ".")
            print("\t\t", answer)
    
            if model not in result:
                result[model] = []
            result[model].append({
                "number": number,
                "answer": answer,
            })
    
            output_file = f"answer.{model}.txt"
            with open(output_file, 'a', encoding='utf-8') as f:
                f.write(f"{number}. {answer}\n")
    
        except Exception as e:
            print(f"Exception: {e}")


# 2

In [7]:
TEMPLATE = """
Ты — профессиональный переводчик с китайского на русский язык.

Исходные данные: список китайских слов в формате JSON:
{
    "word_number": ..., 
    "word_foreign": "...", 
    "transcription": ..., 
    "translation_1": ..., 
    "translation_2": ..., 
    "translation_3": ..., 
    "translation_4": ..., 
    "translation_5": ...
} 

Требования:
1. Для каждого слова:
   - Просмотри все варианты переводов ("translation_1"..."translation_5").
   - На их основе составь один более качественный и точный перевод.
   - Укажи все основные значения, разделяя их запятыми.
   - Перевод должен быть средней длины, не более 20 слов (допустимо чуть больше для многозначных слов).
2. Не добавляй ничего, кроме перевода.
3. Не используй нумерацию внутри перевода, кавычки, скобки, пояснения или примеры.
4. Формат ответа строго:
<word_number>. <word_foreign>: <русский перевод>
5. Не добавляй вводных фраз, заключений или комментариев. Отвечай только в указанном формате.
6. Не изменяй порядок слов и номеров. Для каждого входного слова — один выходной перевод.

Если не можешь точно перевести, всё равно выдай ответ в указанном формате, используя максимально близкое значение.

Входные данные:
{words_json}
"""

TEMPLATE

'\nТы — профессиональный переводчик с китайского на русский язык.\n\nИсходные данные: список китайских слов в формате JSON:\n{\n    "word_number": ..., \n    "word_foreign": "...", \n    "transcription": ..., \n    "translation_1": ..., \n    "translation_2": ..., \n    "translation_3": ..., \n    "translation_4": ..., \n    "translation_5": ...\n} \n\nТребования:\n1. Для каждого слова:\n   - Просмотри все варианты переводов ("translation_1"..."translation_5").\n   - На их основе составь один более качественный и точный перевод.\n   - Укажи все основные значения, разделяя их запятыми.\n   - Перевод должен быть средней длины, не более 20 слов (допустимо чуть больше для многозначных слов).\n2. Не добавляй ничего, кроме перевода.\n3. Не используй нумерацию внутри перевода, кавычки, скобки, пояснения или примеры.\n4. Формат ответа строго:\n<word_number>. <word_foreign>: <русский перевод>\n5. Не добавляй вводных фраз, заключений или комментариев. Отвечай только в указанном формате.\n6. Не и

In [8]:
INPUT_FILE = "./words_Китайский_20250805_194526.json.txt_imported.json"

INPUT_FILE

'./words_Китайский_20250805_194526.json.txt_imported.json'

In [9]:
INDEX_BEGIN = 0
INDEX_END = 50

INDEX_BEGIN, INDEX_END

(0, 50)

In [ ]:
import json

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

words = data['words'][INDEX_BEGIN:INDEX_END]

print(words[0])
print(words[-1])


{
            "translation_4": "один, первый, единый, едино",
